# Show-o2: component gradient cosine

Compare understanding and generation loss gradients on the same attention and FFN parameters in each decoder layer. No optimizer, MLP predictors, pruning, or weight updates.

Use the original token experiment's held-out split: four U examples and two G examples. Compare all eight U/G combinations; these are cross-task comparisons, not matched scenes. Reuse saved U image latents and G reference latents, noise, and all 49 recorded interpolation times. Generation uses the original self-generated pseudo-target flow loss, not a ground-truth image loss.

Run the steps in order in a fresh Colab A100 GPU runtime with high host RAM. GPU memory and runtime for parameter gradients have not been measured. Push this notebook to the configured branch before opening it in Colab.

## 1. Install and clone


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.46.3", "tokenizers==0.20.3", "diffusers==0.30.1",
    "accelerate", "einops", "omegaconf", "timm", "decord", "torchdiffeq",
    "sentencepiece", "Pillow", "numpy", "pandas", "matplotlib",
], check=True)

REPO_ROOT = Path("/content/FlashU-Lab")
REPO_BRANCH = "feature/pilot-acceleration-experiment"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
    "https://github.com/Nicerova7/FlashU-Lab.git", str(REPO_ROOT),
], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))


## 2. Imports and Drive paths

Mount the same Drive used by the token notebook. This experiment reads saved latents, so it does not load the VAE or require the original JPEG paths.


In [ ]:
import gc
import json
import re
from datetime import datetime, timezone
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display
from omegaconf import OmegaConf

from models import Showo2Qwen2_5, omni_attn_mask_naive
from models.misc import get_text_tokenizer, interpolate_pos_encoding
from utils import path_to_llm_name
from evaluation.calculate_token_importance_understanding import assemble_prefill
from evaluation.calculate_token_importance_generation import prepare_generation_inputs
from evaluation.calculate_gradient_update_generation import measured_backbone

drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/intership")
U_NAMES = [
    "counting", "spatial_relations",
    "attribute_combinations", "visual_state_traffic_lights",
]
FOLDERS = {
    f"U/{name}": DATA_ROOT / "gradient_update_understanding" / name
    for name in U_NAMES
}
FOLDERS["G/counting_spatial"] = (
    DATA_ROOT / "gradient_update_generation" / "counting_spatial"
)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR = DATA_ROOT / "component_gradient_cosine" / RUN_ID
OUTPUT_DIR.mkdir(parents=True)
DEVICE = torch.device("cuda:0")
DTYPE = torch.bfloat16


## 3. Select the original held-out examples

Reproduce the token notebook's sorted file order and seed-42 split: 4/1/1 per U category and 8/2/2 for G. Only test examples enter this pilot. No new data or predictor training.


In [ ]:
examples = []
for category, folder in FOLDERS.items():
    paths = sorted(
        path for path in folder.glob("sample_*.pt")
        if re.fullmatch(r"sample_\d+\.pt", path.name)
    )
    for path in paths:
        examples.append(dict(
            uid=f"{category}/{path.stem}", task=category[0],
            category=category, aggregate=str(path),
        ))

rng = np.random.default_rng(42)
for category in [f"U/{name}" for name in U_NAMES]:
    group = [example for example in examples if example["category"] == category]
    for rank, index in enumerate(rng.permutation(len(group))):
        group[index]["split"] = "train" if rank < 4 else "val" if rank == 4 else "test"

group = [example for example in examples if example["task"] == "G"]
for rank, index in enumerate(rng.permutation(len(group))):
    group[index]["split"] = "train" if rank < 8 else "val" if rank < 10 else "test"

manifest = pd.DataFrame(examples)
manifest.to_csv(OUTPUT_DIR / "sample_splits.csv", index=False)
u_examples = [example for example in examples if example["task"] == "U" and example["split"] == "test"]
g_examples = [example for example in examples if example["task"] == "G" and example["split"] == "test"]
display(manifest.groupby(["task", "split"]).size().unstack(fill_value=0))
display(manifest[manifest["split"] == "test"])


## 4. Load Show-o2 and select parameters

Use dense Show-o2 in evaluation mode. Enable differentiation only for Q/K/V/O projections and gate/up/down projections, including their existing biases. Layer norms, embeddings, vision modules, and diffusion heads remain frozen; gradients still propagate through them. Parameters are never updated.


In [ ]:
config = OmegaConf.load(REPO_ROOT / "configs/showo2_1.5b_demo_432x432.yaml")
llm_path = config.model.showo.llm_model_path
tokenizer, token_ids = get_text_tokenizer(
    llm_path, add_showo_tokens=True, return_showo_token_ids=True,
    llm_name=path_to_llm_name[llm_path],
)
model = Showo2Qwen2_5.from_pretrained(
    config.model.showo.pretrained_model_path, use_safetensors=False,
).to(device=DEVICE, dtype=DTYPE).eval().requires_grad_(False)
core = model.showo.model
core.config.use_cache = False
resources = SimpleNamespace(
    model=model, tokenizer=tokenizer, token_ids=token_ids,
    device=DEVICE, dtype=DTYPE,
)

groups = []
for layer_id, layer in enumerate(core.layers):
    for component, module, projections in [
        ("attention", layer.self_attn, ("q_proj", "k_proj", "v_proj", "o_proj")),
        ("ffn", layer.mlp, ("gate_proj", "up_proj", "down_proj")),
    ]:
        named = [
            (f"showo.model.layers.{layer_id}.{component}.{projection}.{name}", parameter)
            for projection in projections
            for name, parameter in getattr(module, projection).named_parameters()
        ]
        for name, parameter in named:
            parameter.requires_grad_(True)
        groups.append(dict(
            layer=layer_id, component=component,
            names=[name for name, parameter in named],
            parameters=[parameter for name, parameter in named],
        ))

parameters = [parameter for group in groups for parameter in group["parameters"]]
parameter_groups = [{
    "layer": group["layer"], "component": group["component"],
    "names": group["names"],
    "shapes": [list(parameter.shape) for parameter in group["parameters"]],
    "num_parameters": sum(parameter.numel() for parameter in group["parameters"]),
} for group in groups]
(OUTPUT_DIR / "parameter_groups.json").write_text(
    json.dumps(parameter_groups, indent=2), encoding="utf-8",
)
print(torch.cuda.get_device_name(DEVICE))
print("Selected parameters:", sum(parameter.numel() for parameter in parameters))


## 5. Define the two losses and cosine

U uses teacher-forced cross-entropy on the saved expected answer. G uses conditional velocity MSE with target reference_latents − initial_noise at the saved interpolation times. The backbone wrapper skips vocabulary logits during G; it keeps the same decoder and final norm.

Each cosine includes every selected parameter in one component. Summing matrix-wise dot products and squared norms is equivalent to flattening and concatenating all matrices. Accumulation uses FP32. Zero-norm gradients yield NaN, not an artificial zero cosine; norms and valid counts are saved.


In [ ]:
def understanding_loss(saved):
    record = saved["sample"]["source_record"]
    answer_ids = tokenizer(
        str(record["expected_answer"]).strip(), add_special_tokens=False,
    )["input_ids"]
    embeds, _, _, layout = assemble_prefill(
        model, tokenizer, token_ids, saved["image_latents"],
        record["prompt"],
        int(config.dataset.preprocessing.max_seq_length) - len(answer_ids),
        omni_attn_mask_naive, interpolate_pos_encoding,
    )
    length = embeds.shape[1]
    answer = torch.tensor([answer_ids], device=DEVICE)
    embeds = torch.cat([embeds, core.embed_tokens(answer)], dim=1).detach()
    offset, span = layout["attention_span"]
    modalities = torch.tensor([[[offset, span]]], device=DEVICE)
    mask = omni_attn_mask_naive(
        B=1, LEN=embeds.shape[1], modalities=modalities,
        device=DEVICE, inverted=True,
    ).to(DTYPE)
    hidden = core(
        inputs_embeds=embeds, attention_mask=mask,
        use_cache=False, output_hidden_states=False, return_dict=True,
    ).last_hidden_state
    logits = model.showo.lm_head(hidden[:, length - 1:-1])
    return F.cross_entropy(
        logits.float().reshape(-1, logits.shape[-1]), answer.reshape(-1),
    )


def generation_backbone(*, inputs_embeds, attention_mask, **kwargs):
    hidden = core(
        inputs_embeds=inputs_embeds, attention_mask=attention_mask,
        use_cache=False, output_hidden_states=False, return_dict=True,
    ).last_hidden_state
    return {"logits": None, "hidden_states": (hidden,)}


def generation_loss(inputs, noise, reference, timestep):
    latent = (1.0 - timestep) * noise + timestep * reference
    time = torch.tensor([timestep], device=DEVICE, dtype=latent.dtype)
    recorder = SimpleNamespace(forward=generation_backbone)
    with measured_backbone(model, recorder):
        _, velocity = model(
            text_tokens=inputs["text_tokens"],
            image_latents=latent, t=time,
            attention_mask=inputs["attention_mask"],
            modality_positions=inputs["modality_positions"],
            max_seq_len=inputs["sequence_length"],
            guidance_scale=0.0, device=str(DEVICE),
        )
    return F.mse_loss(velocity.float(), (reference - noise).float())


def component_cosines(u_gradients, g_gradients):
    rows = []
    start = 0
    for group in groups:
        end = start + len(group["parameters"])
        dot = torch.zeros((), device=DEVICE)
        u_squared = torch.zeros((), device=DEVICE)
        g_squared = torch.zeros((), device=DEVICE)
        for u_gradient, g_gradient in zip(u_gradients[start:end], g_gradients[start:end]):
            u = u_gradient.to(device=DEVICE, dtype=torch.float32)
            g = g_gradient.detach().float()
            dot += (u * g).sum()
            u_squared += u.square().sum()
            g_squared += g.square().sum()
        rows.append(dict(
            layer=group["layer"], component=group["component"],
            cosine=(dot / (u_squared.sqrt() * g_squared.sqrt())).item(),
            u_norm=u_squared.sqrt().item(), g_norm=g_squared.sqrt().item(),
        ))
        start = end
    return rows


## 6. Measure all U/G combinations

Keep one U gradient vector on host RAM, then compare it with each G gradient at every saved time. G passes are repeated for each U example to avoid storing full parameter gradients for all examples and steps. All decoder layers are measured in each pass. This pilot performs 4 U backward passes and 4 × 2 × 49 G backward passes.

The saved references are reused, not regenerated. No optimizer or backward-through-sampling is involved. Results are saved after each U/G combination.


In [ ]:
run = dict(
    run_id=RUN_ID,
    repo_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    repo_branch=REPO_BRANCH,
    checkpoint=config.model.showo.pretrained_model_path,
    model_config=dict(model.config),
    backbone_config=core.config.to_dict(),
    config=OmegaConf.to_container(config, resolve=True),
    torch_version=torch.__version__, dtype=str(DTYPE),
    gpu=torch.cuda.get_device_name(DEVICE),
    pairing="all test U x all test G; not scene-matched",
    understanding_loss="expected-answer teacher-forced cross entropy",
    generation_loss="conditional flow MSE with saved self-generated reference and noise",
    generation_time_scope="all saved interpolation times, averaged within each U/G pair",
    num_u=len(u_examples), num_g=len(g_examples),
    u_examples=u_examples, g_examples=g_examples,
    model_weights_updated=False,
)
(OUTPUT_DIR / "run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")
rows = []
loss_rows = []
source_rows = []

for u_example in u_examples:
    u_saved = torch.load(
        u_example["aggregate"], map_location="cpu", weights_only=True, mmap=True,
    )
    loss_u = understanding_loss(u_saved)
    u_loss_value = loss_u.item()
    u_gradients = tuple(
        gradient.detach().cpu()
        for gradient in torch.autograd.grad(loss_u, parameters)
    )
    source_rows.append(dict(
        uid=u_example["uid"], run_id=u_saved["run_id"],
        sample=u_saved["sample"],
    ))
    loss_rows.append(dict(uid=u_example["uid"], task="U", step=-1, loss=u_loss_value))
    del loss_u, u_saved
    gc.collect()

    for g_example in g_examples:
        g_saved = torch.load(
            g_example["aggregate"], map_location="cpu", weights_only=True, mmap=True,
        )
        inputs = prepare_generation_inputs(
            g_saved["sample"]["source_record"], resources, config,
            {"guidance_scale": 0.0},
        )
        noise = g_saved["initial_noise"].to(DEVICE)
        reference = g_saved["reference_latents"].to(DEVICE)
        source_rows.append(dict(
            uid=g_example["uid"], run_id=g_saved["run_id"],
            sample=g_saved["sample"],
            step_indices=list(g_saved["step_indices"]),
            denoising_timesteps=list(g_saved["denoising_timesteps"]),
        ))

        for step, timestep in zip(g_saved["step_indices"], g_saved["denoising_timesteps"]):
            loss_g = generation_loss(inputs, noise, reference, float(timestep))
            g_loss_value = loss_g.item()
            g_gradients = torch.autograd.grad(loss_g, parameters)
            for result in component_cosines(u_gradients, g_gradients):
                rows.append(dict(
                    u_uid=u_example["uid"], g_uid=g_example["uid"],
                    step=int(step), timestep=float(timestep),
                    u_loss=u_loss_value, g_loss=g_loss_value, **result,
                ))
            loss_rows.append(dict(
                uid=g_example["uid"], u_uid=u_example["uid"],
                task="G", step=int(step), loss=g_loss_value,
            ))
            del loss_g, g_gradients

        pd.DataFrame(rows).to_csv(OUTPUT_DIR / "cosine_per_pair_step.csv", index=False)
        pd.DataFrame(loss_rows).to_csv(OUTPUT_DIR / "losses.csv", index=False)
        (OUTPUT_DIR / "source_samples.json").write_text(
            json.dumps(source_rows, indent=2), encoding="utf-8",
        )
        print(u_example["uid"], "vs", g_example["uid"], "saved", flush=True)
        del g_saved, inputs, noise, reference
        gc.collect()

    del u_gradients
    gc.collect()
    torch.cuda.empty_cache()


## 7. Summarize variability across examples

Average cosines over recorded times within each U/G pair first. Then average over the opposite task to get one value per U example, or one value per G example. Report mean ± sample standard deviation separately across the four U and two G examples. The eight combinations and 49 times are not independent examples. These are descriptive pilot statistics, not confidence intervals or population-level evidence.


In [ ]:
details = pd.DataFrame(rows)
pair_summary = (
    details.groupby(["u_uid", "g_uid", "layer", "component"], as_index=False)
    .agg(cosine=("cosine", "mean"), valid_steps=("cosine", "count"),
         total_steps=("step", "size"))
)
pair_summary.to_csv(OUTPUT_DIR / "cosine_per_pair.csv", index=False)

per_u = (
    pair_summary.groupby(["u_uid", "layer", "component"], as_index=False)
    .agg(cosine=("cosine", "mean"), valid_partners=("cosine", "count"))
    .rename(columns={"u_uid": "uid"})
    .assign(example_task="U")
)
per_g = (
    pair_summary.groupby(["g_uid", "layer", "component"], as_index=False)
    .agg(cosine=("cosine", "mean"), valid_partners=("cosine", "count"))
    .rename(columns={"g_uid": "uid"})
    .assign(example_task="G")
)
per_example = pd.concat([per_u, per_g], ignore_index=True)
summary = (
    per_example.groupby(["example_task", "layer", "component"], as_index=False)
    .agg(mean=("cosine", "mean"), std=("cosine", "std"), n=("cosine", "count"))
)
per_example.to_csv(OUTPUT_DIR / "cosine_per_example.csv", index=False)
summary.to_csv(OUTPUT_DIR / "cosine_mean_std.csv", index=False)
display(summary)


## 8. Plot attention and FFN by layer

Each thin line represents one example after averaging over partners and recorded times. Shading is ±1 standard deviation across examples, not an uncertainty interval. Negative cosines indicate opposing local parameter gradients under these losses; positive cosines indicate alignment. This complements the token-overlap analysis and does not assume an outcome.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
colors = {"attention": "#0072B2", "ffn": "#D55E00"}

for ax, task in zip(axes, ["U", "G"]):
    for component, color in colors.items():
        values = summary[
            (summary["example_task"] == task) & (summary["component"] == component)
        ]
        example_values = per_example[
            (per_example["example_task"] == task) & (per_example["component"] == component)
        ]
        for uid, curve in example_values.groupby("uid"):
            ax.plot(curve["layer"], curve["cosine"], color=color, alpha=0.2, linewidth=0.8)
        ax.plot(values["layer"], values["mean"], color=color, label=component)
        ax.fill_between(
            values["layer"].to_numpy(),
            (values["mean"] - values["std"]).to_numpy(),
            (values["mean"] + values["std"]).to_numpy(),
            color=color, alpha=0.15,
        )
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set(title=f"Variability across {task} examples", xlabel="Decoder layer", ylim=(-1, 1))
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.2)

axes[0].set_ylabel("U–G parameter-gradient cosine")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "component_gradient_cosine.png", dpi=200)
fig.savefig(OUTPUT_DIR / "component_gradient_cosine.pdf")
plt.show()
print("Results:", OUTPUT_DIR)
